# Evaluation reproducibility notebook

Reproduces the three evaluation tables in `Evaluation.tex` / `Appendix.tex` of the paper. Each one answers a different question about the pipeline:

1. **Table `eval-leaderboard`** — extractor leaderboard on the 100-clause holdout (141 gold PPSes), 7 configurations, P/R/F1 after harmless-difference normalisation. **Used to pick the best extractor model.** We score every candidate against the same gold PPSes and take the model with the highest F1; that is `gemma4:31b` (F1 = 0.912).

2. **Table `eval-verifier-perturbation`** — verifier accuracy and macro-F1 on 100 synthetic perturbation cases (binary view: `inconsistent` + `underspecified` vs `non-conflict`), 6 models. **Used to pick the best verifier model.** Each candidate sees the same 100 cases with known verdicts; the model with the highest accuracy / macro-F1 wins. That is `gemma3:27b` (0.970 / 0.962), the verifier we deploy in the pipeline.

3. **Table `eval-verifier-agreement`** — pairwise and all-3 verdict agreement of three verifiers on 100 real findings sampled from the analysis subset. **Used to show that our architecture is robust to the verifier choice.** Even if you swap the deployed verifier for a different model family, the verdict on the same finding is the same in 84.8% of cases — so the inconsistency claims in §RQ3 are not an artefact of one specific model.

Inputs are read from `data/raw/benchmarks/` (extracted from `data/dataset.tar.gz` on first run). The notebook ends with a sanity-check cell that prints every reproduced number alongside the paper value.

## 1. Setup

In [1]:
import json, tarfile
from collections import Counter
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR  = REPO_ROOT / 'data' / 'raw'
BENCH_DIR = DATA_DIR / 'benchmarks'
BUNDLE    = REPO_ROOT / 'data' / 'dataset.tar.gz'

DATA_DIR.mkdir(parents=True, exist_ok=True)
if (BENCH_DIR / 'eval_leaderboard_v3.json').exists():
    print(f'Already extracted at {DATA_DIR}')
else:
    assert BUNDLE.exists(), f'Missing {BUNDLE}'
    print(f'Extracting {BUNDLE.name} ...')
    with tarfile.open(BUNDLE, 'r:gz') as tf:
        tf.extractall(DATA_DIR)
    print('Done.')

leaderboard = json.load(open(BENCH_DIR / 'eval_leaderboard_v3.json'))
perturbation = json.load(open(BENCH_DIR / 'eval_perturbation_v3.json'))
agreement = json.load(open(BENCH_DIR / 'eval_verdict_agreement_v7.json'))
print(f'leaderboard runs : {len(leaderboard["runs"]):>3}')
print(f'perturbation models: {len(perturbation["models"]):>3}')
print(f'agreement models : {len(agreement):>3}')

Already extracted at /srv/lustre01/project/vr_outsec-vh2sz1t4fks/users/soufiane.essahli/When-Policies-Disagree/data/raw
leaderboard runs :  16
perturbation models:   6
agreement models :   3


## 2. Extractor leaderboard (Table `eval-leaderboard`)

**Purpose: choose the best extractor model.** Every candidate runs over the same 100-clause holdout and is scored against the same 141 gold Privacy Practice Statements. The model with the highest F1 becomes the production extractor.

Strict matches each predicted PPS to a gold PPS using the bipartite matcher of prior work. Adjusted normalises two harmless differences before counting: same-action splits at different data-object granularity, and non-schema actions that map cleanly to a schema action (`rent`, `trade`, `disclose`, ...). The paper reports **adjusted** P/R/F1.

Result: `gemma4:31b` wins with F1 = 0.912 (P 0.871, R 0.957). Seven configurations from the appendix table:

In [2]:
_PAPER_CONFIGS = [
    ('gemma4_31b__reflect_off',           'gemma4:31b'),
    ('gpt_oss_120b__reflect_off',         'gpt-oss:120b'),
    ('qwen3_vl_instruct__fewshot',        'qwen3-vl-instruct (few-shot)'),
    ('qwen3_vl_instruct__reflect_off',    'qwen3-vl-instruct'),
    ('deepseek_v4_flash__reflect_off',    'deepseek-v4-flash'),
    ('gemma3_27b__reflect_off',           'gemma3:27b'),
    ('qwen3_next_80b__reflect_off',       'qwen3-next:80b'),
]
by_run = {r['run_key']: r for r in leaderboard['runs']}
rows = []
for run_key, label in _PAPER_CONFIGS:
    r = by_run.get(run_key)
    if not r:
        continue
    a = r['adjusted']
    rows.append((label, a['precision'], a['recall'], a['f1']))
rows.sort(key=lambda t: -t[3])

print(f'Holdout: {leaderboard["n_clauses"]} clauses, {leaderboard["n_gold_pps"]} gold PPSes')
print()
print(f'{"Configuration":<32s}  {"Precision":>9s}  {"Recall":>7s}  {"F1":>7s}')
print('-' * 60)
for label, p, r, f in rows:
    print(f'{label:<32s}  {p:>9.3f}  {r:>7.3f}  {f:>7.3f}')

Holdout: 100 clauses, 141 gold PPSes

Configuration                     Precision   Recall       F1
------------------------------------------------------------
gemma4:31b                            0.871    0.957    0.912
gpt-oss:120b                          0.878    0.915    0.896
qwen3-vl-instruct (few-shot)          0.886    0.848    0.867
qwen3-vl-instruct                     0.934    0.712    0.808
deepseek-v4-flash                     0.844    0.766    0.803
gemma3:27b                            0.310    0.900    0.462
qwen3-next:80b                        0.950    0.270    0.420


## 3. Verifier perturbation (Table `eval-verifier-perturbation`)

**Purpose: choose the best verifier model.** Real-world inconsistencies are scarce and expensive to label, so we build 100 synthetic clause pairs with known verdicts and use them as a benchmark. Each case is constructed by perturbing a real PPS in one of four ways — flipping a modality, changing a condition, changing a purpose, or making a retention window incompatible — plus unperturbed compatible pairs as `non-conflict` controls. Mix: 18 inconsistent, 56 underspecified, 26 non-conflict.

We then run every candidate verifier on the same 100 cases and pick the model with the highest accuracy / macro-F1. Since `inconsistent` and `underspecified` collapse to a single "genuine finding" class downstream, the table reports the **binary view** (genuine vs non-conflict).

Result: `gemma3:27b` wins at 0.970 accuracy / 0.962 macro-F1; the other five drop to 0.48–0.82 because they often mistake genuine inconsistencies for non-conflict. `gemma3:27b` is the verifier deployed in the pipeline.

In [3]:
gt = perturbation['ground_truth_distribution']
print(f'Cases: {sum(gt.values())} (mix: {gt})')
print()
rows = sorted(
    [(m, v['accuracy_binary'], v['macro_f1_binary']) for m, v in perturbation['models'].items()],
    key=lambda t: -t[1],
)
print(f'{"Model":<22s}  {"Accuracy":>9s}  {"Macro F1":>9s}')
print('-' * 45)
for model, acc, mf1 in rows:
    print(f'{model:<22s}  {acc:>9.3f}  {mf1:>9.3f}')

Cases: 100 (mix: {'soft_tension': 56, 'hard_contradiction': 18, 'non_conflict': 26})

Model                    Accuracy   Macro F1
---------------------------------------------
gemma3-27b                  0.970      0.962
deepseek-v4-flash           0.820      0.802
gpt-oss-120b                0.770      0.755
gemma-4-31b                 0.760      0.745
qwen3-vl-instruct           0.660      0.651
qwen3-next-80b              0.480      0.479


## 4. Verifier agreement (Table `eval-verifier-agreement`)

**Purpose: show that the architecture is robust to the verifier choice.** The perturbation experiment picks one model on synthetic cases. This table shows that when you swap the deployed verifier for a different model family on **real** findings, you get mostly the same answer — so §RQ3's inconsistency claims are not an artefact of one specific verifier.

We sample 100 `inconsistent` candidates from the analysis subset (seed 42), feed each one to three verifier models from different families (`deepseek-v3.1:671b`, `gpt-oss:120b`, `gemma4:31b`), and check how often they pick the same verdict.

Result: `deepseek-v3.1:671b` and `gemma4:31b` agree on every finding (100%); `gpt-oss:120b` is the only outlier, and even there agreement stays at 84.8%. **All three verifiers assign the same verdict to 84.8% of findings**, indicating the verifier output is stable across model families — swapping verifiers does not change the qualitative findings.

In [4]:
models = list(agreement.keys())
ids = sorted({i for v in agreement.values() for i in v.keys()})
n = len(ids)
print(f'Cases: {n} | models: {models}')
print()
_NICE = {
    'deepseek-v31-671b': 'deepseek-v3.1:671b',
    'gemma4-31b':        'gemma4:31b',
    'gpt-oss-120b':      'gpt-oss:120b',
}
pairwise = []
for i, m1 in enumerate(models):
    for m2 in models[i+1:]:
        agree = sum(1 for x in ids if agreement[m1][x] == agreement[m2][x])
        pairwise.append((m1, m2, agree, n))
pairwise.sort(key=lambda t: -t[2])
print(f'{"Model A":<22s}  {"Model B":<22s}  {"Agreement":>10s}')
print('-' * 60)
for m1, m2, a, total in pairwise:
    print(f'{_NICE.get(m1, m1):<22s}  {_NICE.get(m2, m2):<22s}  {100*a/total:>9.1f}%')
unan = sum(1 for x in ids if len({agreement[m][x] for m in models}) == 1)
print()
print(f'All-{len(models)} unanimous: {unan}/{n} = {100*unan/n:.1f}%')

Cases: 33 | models: ['deepseek-v31-671b', 'gpt-oss-120b', 'gemma4-31b']

Model A                 Model B                  Agreement
------------------------------------------------------------
deepseek-v3.1:671b      gemma4:31b                  100.0%
deepseek-v3.1:671b      gpt-oss:120b                 84.8%
gpt-oss:120b            gemma4:31b                   84.8%

All-3 unanimous: 28/33 = 84.8%


## 5. Sanity check vs paper

In [5]:
checks = []

# Leaderboard (Table eval-leaderboard) -- paper-quoted P/R/F1 (adjusted)
_PAPER_LB = {
    'gemma4_31b__reflect_off':         {'precision': 0.871, 'recall': 0.957, 'f1': 0.912},
    'gpt_oss_120b__reflect_off':       {'precision': 0.878, 'recall': 0.915, 'f1': 0.896},
    'qwen3_vl_instruct__fewshot':      {'precision': 0.886, 'recall': 0.848, 'f1': 0.867},
    'qwen3_vl_instruct__reflect_off':  {'precision': 0.934, 'recall': 0.712, 'f1': 0.808},
    'deepseek_v4_flash__reflect_off':  {'precision': 0.844, 'recall': 0.766, 'f1': 0.803},
    'gemma3_27b__reflect_off':         {'precision': 0.310, 'recall': 0.900, 'f1': 0.462},
    'qwen3_next_80b__reflect_off':     {'precision': 0.950, 'recall': 0.270, 'f1': 0.420},
}
for run_key, expected in _PAPER_LB.items():
    a = by_run[run_key]['adjusted']
    for metric, exp in expected.items():
        got = round(a[metric], 3)
        checks.append((f'leaderboard {run_key} {metric}', got, exp))

# Perturbation (Table eval-verifier-perturbation) -- paper binary view
_PAPER_PERT = {
    'gemma3-27b':         {'acc': 0.970, 'mf1': 0.962},
    'deepseek-v4-flash':  {'acc': 0.820, 'mf1': 0.802},
    'gpt-oss-120b':       {'acc': 0.770, 'mf1': 0.755},
    'gemma-4-31b':        {'acc': 0.760, 'mf1': 0.745},
    'qwen3-vl-instruct':  {'acc': 0.660, 'mf1': 0.651},
    'qwen3-next-80b':     {'acc': 0.480, 'mf1': 0.479},
}
for model, exp in _PAPER_PERT.items():
    s = perturbation['models'][model]
    checks.append((f'perturbation {model} accuracy_binary', s['accuracy_binary'], exp['acc']))
    checks.append((f'perturbation {model} macro_f1_binary', s['macro_f1_binary'], exp['mf1']))

# Agreement (Table eval-verifier-agreement) -- paper pairwise + all-3 unanimous
_PAPER_AGREE = {
    ('deepseek-v31-671b', 'gemma4-31b'):    1.000,
    ('deepseek-v31-671b', 'gpt-oss-120b'):  0.848,
    ('gpt-oss-120b',      'gemma4-31b'):    0.848,
}
for (m1, m2), exp in _PAPER_AGREE.items():
    a, b = (m1, m2) if m1 in agreement else (m2, m1)
    agree = sum(1 for x in ids if agreement[a][x] == agreement[b][x]) / n
    checks.append((f'agreement {m1} vs {m2}', round(agree, 3), exp))
checks.append(('agreement all-3 unanimous', round(unan / n, 3), 0.848))

ok = True
for k, got, expected in checks:
    match = abs(got - expected) < 0.01
    print(f'  {k:<60s}  got={got:<7}  paper={expected:<7}  {"OK" if match else "MISMATCH"}')
    if not match:
        ok = False
print('\n' + ('PASSED' if ok else 'FAILED'))

  leaderboard gemma4_31b__reflect_off precision                 got=0.871    paper=0.871    OK
  leaderboard gemma4_31b__reflect_off recall                    got=0.957    paper=0.957    OK
  leaderboard gemma4_31b__reflect_off f1                        got=0.912    paper=0.912    OK
  leaderboard gpt_oss_120b__reflect_off precision               got=0.878    paper=0.878    OK
  leaderboard gpt_oss_120b__reflect_off recall                  got=0.915    paper=0.915    OK
  leaderboard gpt_oss_120b__reflect_off f1                      got=0.896    paper=0.896    OK
  leaderboard qwen3_vl_instruct__fewshot precision              got=0.886    paper=0.886    OK
  leaderboard qwen3_vl_instruct__fewshot recall                 got=0.848    paper=0.848    OK
  leaderboard qwen3_vl_instruct__fewshot f1                     got=0.867    paper=0.867    OK
  leaderboard qwen3_vl_instruct__reflect_off precision          got=0.934    paper=0.934    OK
  leaderboard qwen3_vl_instruct__reflect_off recal